<a href="https://colab.research.google.com/github/RogMaverick18/Speech-Processing-Lab-assignments/blob/main/Lab%204%20-%20task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 4: Discussion
**Lab:** Frame-wise Analysis of a Speech Signal — Time-Domain Features

## (a) Voiced vs Unvoiced Speech: Energy, ZCR, and Periodicity

### Short-Time Energy (STE)
STE measures the total power within a frame: $E(m) = \sum_{n} x^2(n)$.

- **Voiced speech** (vowels, nasals, voiced fricatives) is produced by periodic vocal-fold vibration. The glottis opens and closes at the fundamental frequency $F_0$, releasing bursts of air that excite the vocal tract resonances. This yields large, sustained amplitude oscillations → **high STE**.
- **Unvoiced speech** (stops, fricatives, affricates) is produced by turbulent airflow through a constriction with the glottis open (no vibration). The resulting signal is noise-like with low amplitude → **low STE**.
- **Silence/pauses** have near-zero STE.

STE is therefore a reliable indicator of speech activity and of voiced vs unvoiced regions, though it is sensitive to absolute recording level.

### Zero-Crossing Rate (ZCR)
ZCR counts the number of times the signal changes sign per frame: $Z(m) = \frac{1}{2L}\sum_{n}|\text{sgn}[x(n)] - \text{sgn}[x(n-1)]|$.

- **Voiced frames** have a nearly sinusoidal waveform at $F_0$ (typically 80–300 Hz). At 16 kHz sampling, such a signal crosses zero only ~160–600 times per second, giving a **low ZCR** per 25 ms frame (≈ 2–15 crossings).
- **Unvoiced frames** contain broadband noise energy concentrated at higher frequencies; the signal oscillates rapidly → **high ZCR** (often 3–10× that of voiced).

The complementary behaviour of STE and ZCR is the basis of classic V/UV detectors: a frame is classified as voiced when STE is high **and** ZCR is low, and as unvoiced when STE is low and ZCR is high.

### Periodicity
- Voiced speech is **quasi-periodic** at the pitch period $T_0 = 1/F_0$. This periodicity is visible as repeating peaks in autocorrelation and deep minima in AMDF/AMSDF.
- Unvoiced speech is **aperiodic**; its autocorrelation decays immediately and AMDF/AMSDF show no organised minimum structure.
- Mixed frames (e.g., voiced fricatives) show moderate energy and intermediate ZCR, with a weaker periodic component.

## (b) How Periodicity-Based Features Help in Pitch Detection

The fundamental frequency (pitch) of voiced speech represents the repetition rate of vocal-fold vibration. Periodicity-based features detect this repetition directly in the time domain.

### Autocorrelation Method
$$r(k) = \sum_{n=0}^{N-k-1} x(n)\,x(n+k)$$

For a periodic signal with period $T_0$, $x(n) = x(n + T_0)$, so $r(T_0) \approx r(0)$. A large peak appears at lag $k = T_0$ (and its multiples). The pitch frequency is then:
$$F_0 = \frac{F_s}{\hat{k}}$$
where $\hat{k}$ is the lag of the first prominent peak beyond lag 0.

### AMDF Method
$$\text{AMDF}(k) = \frac{1}{N-k}\sum_{n=0}^{N-k-1}|x(n) - x(n+k)|$$

At lag $k = T_0$, the signal aligns with itself → differences are near zero → a **minimum** appears. The pitch period is the lag of the first significant minimum in the valid search range.

### AMSDF Method
$$\text{AMSDF}(k) = \frac{1}{N-k}\sum_{n=0}^{N-k-1}[x(n) - x(n+k)]^2$$

Squaring amplifies small differences and suppresses large ones non-linearly, making the minimum at $k = T_0$ **sharper and easier to locate** compared to AMDF.

### Why these features are effective
1. They operate **directly on the time-domain samples** — no spectral transformation is needed, keeping computation fast.
2. They are inherently **template-matching** approaches: the pitch period is found wherever the frame best matches a delayed copy of itself.
3. Restricting the search to physically plausible pitch lags (50–500 Hz, i.e., lags 32–320 at 16 kHz) eliminates spurious detections.

## (c) Reliability of Each Time-Domain Feature for Pitch Detection

| Feature | Reliability for Pitch Detection | Strengths | Weaknesses |
|---|---|---|---|
| **STE** | ✗ Not a pitch estimator | Detects voiced activity | Provides no frequency information; cannot estimate $F_0$ |
| **STM** | ✗ Not a pitch estimator | Amplitude envelope tracking | Same as STE; no periodicity information |
| **ZCR** | ⚠ Poor | Simple; noise-robust for V/UV decision | Only reliable for pure sinusoids; greatly affected by noise and harmonics; cannot give an accurate $F_0$ |
| **Autocorrelation** | ✓✓ Best overall | Robust to additive noise; well-understood; works across a wide SNR range; picks up fundamental even when harmonics dominate | Can double/halve pitch if the second peak is stronger (octave error); computationally $O(N^2)$ but can use FFT |
| **AMDF** | ✓ Good | No multiplications (only subtractions and absolute values) → very fast; similar accuracy to autocorrelation for clean speech | More sensitive to additive noise than autocorrelation; minimum can be ambiguous when multiple harmonics are present |
| **AMSDF** | ✓ Good–Very Good | Sharper minimum than AMDF (squaring amplifies contrast); precise for clean speech | Squaring also amplifies noise, making it less robust in noisy conditions; same octave-error susceptibility as autocorrelation |

### Overall conclusion
- For **pitch estimation**, the recommended hierarchy is: **Autocorrelation ≥ AMSDF > AMDF ≫ ZCR**.
- STE and STM are best used as **pre-selectors** — they identify which frames are voiced and should even be processed for pitch.
- In practice, robust pitch trackers (e.g., YIN, RAPT) combine autocorrelation with post-processing (peak selection, median smoothing, inter-frame continuity) to handle noise, unvoiced frames, and octave errors.